# 10 Figures
Publication figures, drawn only from files the earlier notebooks saved: the cached anatomy and tractograms of each
session (`data/cache/`) and the result tables (`outputs/results/`). Nothing is tracked, trained or scored here, so
the notebook can be re-run at any time to restyle the figures. Each figure is saved as PDF (vector, for the journal)
and PNG in `outputs/figures/`.

In [ ]:
DEV_MODE = True    # True: small test run (dev: block of config.yaml). False: full experiment (full: block).

import json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.collections import LineCollection, PolyCollection
from matplotlib.patches import Patch
from retinopath.core import load_config
from retinopath import metrics as M, pipeline as PL

cfg = load_config(DEV_MODE)
RES = cfg.P.out / "results"; FIG = cfg.P.out / "figures"; FIG.mkdir(exist_ok=True)
print(f"DEV_MODE={DEV_MODE} | outputs -> {cfg.P.out}")
if DEV_MODE: print("DEV_MODE: small test run. Its outputs are kept apart in outputs/dev and are not experiment results.")

## Shared settings
Subjects, protocols, colours and the loaders used by every figure. A network is one method per variant: its
training seeds are averaged in the tables and pooled in the streamline counts.

In [ ]:
plt.rcParams.update({"font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8, "legend.fontsize": 7, "xtick.labelsize": 7,
                     "ytick.labelsize": 7, "axes.spines.top": False, "axes.spines.right": False, "pdf.fonttype": 42, "svg.fonttype": "none"})
subjects = json.load(open(cfg.P.out / "subjects.json")); test = subjects["test"]
clin = [p for p in cfg.clinical_protocols if p in cfg.run_protocols]; MAIN = clin[0]      # protocol of the main panels
PAIR_NAMES = list(M.PAIRS.values())
STYLE = {"reference": ("k", "reference (msmt_csd, full data)"), "proximity": ("0.7", "proximity only"), "csd": ("0.45", "classical CSD"),
         "voxel": ("C0", "network, voxel loss"), "path": ("C3", "network, path loss")}
OTHER = ["#e69f00", "#56b4e9", "#009e73", "#cc79a7", "#8c564b", "#bcbd22", "#17becf", "#9467bd", "#000080", "#f0e442"]
_extra = {}
def COL(m):
    """Fixed colour per method: the five main ones from STYLE, the others in order of first use."""
    if m in STYLE: return STYLE[m][0]
    return _extra.setdefault(m, OTHER[len(_extra) % len(OTHER)])

def variant(method):
    """'net_path_s0' -> 'path'; classical names are kept."""
    return method[4:].rsplit("_s", 1)[0] if method.startswith("net_") else method

def table(name):
    f = RES / f"{name}.csv"
    if not f.exists(): return pd.DataFrame()
    d = pd.read_csv(f); d["variant"] = d.method.map(variant) if "method" in d else None; return d

def anat_of(sid, kind="3T"):
    z = np.load(cfg.P.cache / PL.session(sid, kind)["tag"] / "anat.npz")
    return {k: z[k].item() if z[k].ndim == 0 else z[k] for k in z.files}

def tracts(sid, name, method, kind="3T"):
    """Cached tractograms of one method; for a network variant, those of every training seed. Never computes."""
    d = cfg.P.cache / PL.session(sid, kind)["tag"]
    files = sorted(d.glob(f"tract_{name}_net_{method}_s*.npz")) if method in STYLE and method not in ("reference", "proximity", "csd") \
        else [d / f"tract_{name}_{method}.npz"]
    return [dict(np.load(f)) for f in files if f.exists()]

def save(fig, name):
    for ext in ("pdf", "png"): fig.savefig(FIG / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    print("saved", FIG / f"{name}.pdf")

def letter(ax, s, x=-0.12):
    ax.text(x, 1.06, s, transform=ax.transAxes, fontsize=10, fontweight="bold", va="bottom")

ref, prot, lea = table("h1_reference"), table("protocols"), table("learned")
print("test subjects:", len(test), "| clinical protocols:", clin, "| main panels:", MAIN)

## Like-to-like counts
For one area pair, streamline ends are binned by the retinotopic coordinate of each end (eccentricity or polar
angle), with bin edges at quantiles of the subject's valid V1 values (`prf.n_ecc_bins` bins, as for the binned AUC).
Each count is divided by what independent ends would give (row total × column total / total). A ratio above 1 on
the diagonal means ends at matching visual-field positions are joined more often than chance. This ratio removes
uneven coverage but not distance; the distance-matched RSI is the controlled measure.

In [ ]:
def binned_counts(anat, key, t, code, var):
    """Observed and expected (independent ends) counts, bins of `var` ('ecc' or 'ang') of end a x end b."""
    nb = cfg.prf.n_ecc_bins; v = key[var]
    ok = key["valid"] & (anat["area"] == 1)
    if ok.sum() < nb: return np.zeros((nb, nb)), np.zeros((nb, nb))
    edges = np.quantile(v[ok], np.linspace(0, 1, nb + 1))
    m = (t["pair"] == code) & key["valid"][t["a"]] & key["valid"][t["b"]]
    ba, bb = (np.clip(np.digitize(v[t[e][m]], edges[1:-1]), 0, nb - 1) for e in ("a", "b"))
    O = np.zeros((nb, nb)); np.add.at(O, (ba, bb), 1)
    E = np.outer(O.sum(1), O.sum(0)) / max(O.sum(), 1)
    return O, E

def pooled(method, name, code, var, sids=test, kind="3T"):
    """Counts summed over subjects (and training seeds) -> ratio observed / expected; per-subject ratio by bin distance."""
    O = E = 0; prof = []
    for sid in sids:
        a = anat_of(sid, kind); key = M.get_key(a, PL.cfg_for(cfg, PL.session(sid, kind)))
        o = e = 0
        for t in tracts(sid, name, method, kind):
            oo, ee = binned_counts(a, key, t, code, var); o, e = o + oo, e + ee
        if np.sum(o) == 0: continue
        O, E = O + o, E + e
        i, j = np.indices(o.shape); d = np.abs(i - j)
        prof.append([o[d == k].sum() / e[d == k].sum() if e[d == k].sum() else np.nan for k in range(o.shape[0])])
    R = np.where(np.asarray(E) > 0, np.asarray(O) / np.where(np.asarray(E) > 0, E, 1), np.nan) if np.ndim(O) else None
    return R, np.array(prof)

def matrix(ax, R, title, pair, var, vmax=1.0):
    nb = cfg.prf.n_ecc_bins
    if R is None:
        ax.text(0.5, 0.5, "no streamlines", ha="center", va="center", transform=ax.transAxes); ax.set_axis_off(); return None
    ax.set_facecolor("0.9")      # grey: no streamline end in that row or column
    im = ax.imshow(np.log2(np.maximum(R, 2.0 ** -vmax)), origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    a1, a2 = pair.split("-"); lab = "eccentricity" if var == "ecc" else "polar angle"
    ax.set_xticks(range(nb), [str(k + 1) for k in range(nb)]); ax.set_yticks(range(nb), [str(k + 1) for k in range(nb)])
    ax.set_xlabel(f"{a2} {lab} bin"); ax.set_ylabel(f"{a1} {lab} bin"); ax.set_title(title)
    return im

def colorbar(fig, im, ax):
    """Colour bar beside a matrix panel, exactly as tall as the matrix."""
    cb = fig.colorbar(im, cax=ax.inset_axes([1.08, 0, 0.07, 1]), ticks=[-1, 0, 1]); cb.ax.set_yticklabels(["½×", "chance", "2×"]); cb.set_label("streamlines / expected")

## Figure 1: retinotopy and V1–V3 tractography of one subject
The first test subject, first hemisphere, seen from the medial side. (a) Visual areas, (b) polar angle and (c)
eccentricity from the subject's own pRF fit, on the very inflated surface: the whole hemisphere in (a), the boxed region
enlarged in (b, c). Without that file the white surface near V1–V3 is drawn instead. Vertices outside the pRF validity mask are dark grey. (d) The reference V1–V3 streamlines on the
white surface, coloured by the eccentricity of their lower-area end (V1, or V2 for V2–V3).

In [ ]:
from retinopath.subject import inflated_surface
sid = test[0]; a = anat_of(sid); key = M.get_key(a, cfg); h = 0; hemi = a["hemis"][h]
roi = (a["area"] > 0) & (a["hemi"] == h); lo, hi = cfg.prf.ecc_min, cfg.prf.ecc_max
def camera(deg):
    """Unit vector towards the viewer, on the medial side, turned `deg` towards the back; and the screen's x axis."""
    v = np.array([np.cos(np.radians(deg)) * (1 if hemi == "L" else -1), -np.sin(np.radians(deg)), 0.0]); return v, np.array([-v[1], v[0], 0.0])
inf = inflated_surface(cfg, sid, hemi)
Pi = inf[a["vert_id"]] if inf is not None else a["verts"]
if inf is None: print("very inflated surface not downloaded (notebook 01): panels a-c use the white surface")

class View:
    """Orthographic view of the surface P within `radius` mm of V1-V3 (None: whole hemisphere): faces sorted back
    to front, Lambert shading."""
    def __init__(self, P, radius, deg):
        d, self.right = camera(deg)
        near = (a["hemi"] == h) & (True if radius is None else np.linalg.norm(P - P[roi].mean(0), axis=1) < radius)
        F = a["faces"][near[a["faces"]].all(1)]; self.F = F[np.argsort((P @ d)[F].mean(1))]
        n = np.cross(P[self.F[:, 1]] - P[self.F[:, 0]], P[self.F[:, 2]] - P[self.F[:, 0]])
        self.shade = 0.55 + 0.45 * np.abs(n @ d) / (np.linalg.norm(n, axis=1) + 1e-12)
        self.xy = np.c_[P @ self.right, P[:, 2]]; self.box = self.xy[np.unique(self.F)]
    def colors(self, values=None, cmap=None, vmin=0, vmax=1, mask=None):
        F = self.F; c = np.tile([0.82, 0.82, 0.82, 1.0], (len(F), 1)); c[roi[F].all(1), :3] = 0.55
        if values is not None:
            m = mask[F].all(1); c[m] = plt.get_cmap(cmap)(np.clip((values[F[m, 0]] - vmin) / (vmax - vmin), 0, 1))
        c[:, :3] *= self.shade[:, None]; return c
    def zoom_box(self, pad=5.0):
        """(x0, x1, y0, y1) around V1-V3 on screen."""
        q = self.xy[roi]; return q[:, 0].min() - pad, q[:, 0].max() + pad, q[:, 1].min() - pad, q[:, 1].max() + pad
    def draw(self, ax, colors, alpha=1.0, zoom=False):
        colors = colors.copy(); colors[:, 3] = alpha
        ax.add_collection(PolyCollection(self.xy[self.F], facecolors=colors, edgecolors="none"))
        x0, x1, y0, y1 = self.zoom_box() if zoom else (self.box[:, 0].min() - 2, self.box[:, 0].max() + 2, self.box[:, 1].min() - 2, self.box[:, 1].max() + 2)
        ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
        ax.set_aspect("equal"); ax.set_axis_off()

def cbar(ax, cmap, vmin, vmax, label, ticks):
    cb = plt.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin, vmax)), cax=ax.inset_axes([0.15, -0.08, 0.7, 0.04]),
                      orientation="horizontal", ticks=ticks)
    cb.set_label(label, labelpad=1)

vi = View(Pi, None, 15) if inf is not None else View(Pi, 40, 30); vw = View(a["verts"], 40, 30)
fig, ax = plt.subplots(1, 4, figsize=(7.2, 2.6)); fig.subplots_adjust(wspace=0.08, left=0.03, right=0.99, top=0.9, bottom=0.2)
AREA = ["#1b9e77", "#d95f02", "#7570b3"]
c = vi.colors(); m = roi[vi.F].all(1)
c[m, :3] = np.array([plt.matplotlib.colors.to_rgb(AREA[int(v) - 1]) for v in a["area"][vi.F[m, 0]]]) * vi.shade[m, None]
vi.draw(ax[0], c)
zoom = inf is not None                                            # whole inflated hemisphere in a, V1-V3 enlarged in b and c
if zoom:
    x0, x1, y0, y1 = vi.zoom_box(); ax[0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, lw=0.6, color="k"))
    (l, r), (b, t) = ax[0].get_xlim(), ax[0].get_ylim(); ax[0].set_xlim(min(l, x0 - 1), max(r, x1 + 1)); ax[0].set_ylim(min(b, y0 - 1), max(t, y1 + 1))
ax[0].legend(handles=[Patch(color=AREA[k], label=f"V{k + 1}") for k in range(3)], loc="upper center",
                                 bbox_to_anchor=(0.5, -0.02), ncol=3, frameon=False, handlelength=1, columnspacing=1)
vi.draw(ax[1], vi.colors(np.nan_to_num(key["ang"]), "hsv", 0, 360, key["valid"]), zoom=zoom); cbar(ax[1], "hsv", 0, 360, "polar angle (deg)", [0, 90, 180, 270, 360])
vi.draw(ax[2], vi.colors(np.nan_to_num(key["ecc"]), "viridis", lo, hi, key["valid"]), zoom=zoom); cbar(ax[2], "viridis", lo, hi, "eccentricity (deg)", np.linspace(lo, hi, 3).round(1))
t = tracts(sid, "REF", "msmt_csd")[0]; sl = PL.streamlines_of(t)
keep = [i for i in range(len(sl)) if key["valid"][t["a"][i]] and a["hemi"][t["a"][i]] == h]
keep = np.random.default_rng(cfg.seed).permutation(keep)[:1500]
vw.draw(ax[3], vw.colors(), alpha=0.35)
ax[3].add_collection(LineCollection([np.c_[sl[i] @ vw.right, sl[i][:, 2]] for i in keep], linewidths=0.4,
                                    colors=plt.get_cmap("viridis")(np.clip((key["ecc"][t["a"][keep]] - lo) / (hi - lo), 0, 1))))
cbar(ax[3], "viridis", lo, hi, "eccentricity of lower-area end (deg)", np.linspace(lo, hi, 3).round(1))
for x, s, ttl in zip(ax, "abcd", ("visual areas", "polar angle", "eccentricity", "reference streamlines")):
    x.set_title(ttl); x.text(0.0, 1.02, s, transform=x.transAxes, fontsize=10, fontweight="bold", va="bottom")
save(fig, "fig1_retinotopy_tractography")
print("subject", sid, "hemisphere", hemi, "|", "very inflated" if inf is not None else "white", "surface |", len(keep), "streamlines drawn")

## Figure 2: retinotopy of the reference tractogram (H1)
(a–c) Eccentricity and (d–f) polar-angle like-to-like ratios of the reference tractogram, summed over test subjects.
(g, h) Ratio by distance between bins, one line per subject, reference against the proximity-only tractogram.
(i) RSI per area pair, one point per subject; lines join the same subject.

In [ ]:
fig = plt.figure(figsize=(7.2, 7.4)); gs = fig.add_gridspec(3, 3, hspace=0.6, wspace=0.75)
for r, var in enumerate(("ecc", "ang")):
    for c, (code, pair) in enumerate(M.PAIRS.items()):
        x = fig.add_subplot(gs[r, c]); R, _ = pooled("msmt_csd", "REF", code, var)
        im = matrix(x, R, pair, pair, var); letter(x, "abcdef"[3 * r + c], -0.35)
    if im is not None: colorbar(fig, im, x)
bottom = gs[2, :].subgridspec(1, 3, wspace=0.5)
for c, var in enumerate(("ecc", "ang")):
    x = fig.add_subplot(bottom[c])
    for m, name in (("msmt_csd", "reference"), ("proximity", "proximity")):
        _, pr = pooled(m, "REF", 12, var)
        if len(pr):
            x.plot(pr.T, color=COL(name), lw=0.6, alpha=0.5); x.plot(np.nanmedian(pr, 0), color=COL(name), lw=2, label=STYLE[name][1])
    x.axhline(1, color="k", lw=0.6, ls=":"); x.set_xlabel(f"V1-V2 {'eccentricity' if var == 'ecc' else 'polar-angle'} bin distance")
    x.set_ylabel("streamlines / expected"); letter(x, "gh"[c])
    if c == 0: x.legend(frameon=False, loc="lower center", bbox_to_anchor=(1.1, 1.08), ncol=2)
x = fig.add_subplot(bottom[2])
for k, pair in enumerate(PAIR_NAMES):
    w = ref.pivot_table(index="sid", columns="method", values=f"rsi_ree_{pair}")
    for m, dx in (("msmt_csd", -0.15), ("proximity", 0.15)):
        if m in w: x.scatter(np.full(len(w), k + dx), w[m], s=10, color=COL("reference" if m == "msmt_csd" else m), zorder=3)
    if {"msmt_csd", "proximity"} <= set(w): x.plot([k - 0.15, k + 0.15], w[["msmt_csd", "proximity"]].T, color="0.6", lw=0.5)
x.axhline(0, color="k", lw=0.6); x.set_xticks(range(3), PAIR_NAMES); x.set_ylabel("RSI (visual-field distance)"); letter(x, "i")
save(fig, "fig2_h1_reference")

## Figure 3: V1–V2 streamlines of one subject, by method
The subject, hemisphere and view of Figure 1d; every method starts from the same seeds. Columns: the reference on
the full data, then classical CSD and the two networks on the main clinical-grade protocol. Top row: each V1–V2
streamline coloured by the eccentricity of its V1 end. Bottom row: coloured by its endpoint error, the visual-field
distance between the pRF centres of its two ends, with the colour scale ending at half of `prf.ecc_max`. Each panel gives the number of streamlines drawn and the
subject's V1–V2 RSI for that method.

In [ ]:
def subject_rsi(var, name, col="rsi_ree_V1-V2"):
    """RSI of the Figure 1 subject for one method (network seeds averaged); NaN if it was not scored."""
    tab = ref if name == "REF" else prot if var == "csd" else lea
    d = tab[(tab.sid.astype(str) == str(sid)) & (tab.protocol == name) & (tab.variant == var)][col]
    return d.mean() if len(d) else np.nan

cols = [("REF", "msmt_csd", "reference"), (MAIN, "csd", "csd"), (MAIN, "voxel", "voxel"), (MAIN, "path", "path")]
emax = cfg.prf.ecc_max / 2                                        # top of the endpoint-error colour scale (deg)
rng = np.random.default_rng(cfg.seed)
fig, ax = plt.subplots(2, 4, figsize=(7.2, 4.3)); fig.subplots_adjust(wspace=0.04, hspace=0.08, left=0.05, right=0.9, top=0.92, bottom=0.03)
for c, (name, m, var) in enumerate(cols):
    ts = tracts(sid, name, m); ax[0, c].set_title(STYLE[var][1] + ("" if name == "REF" else f", {name}"), fontsize=7)
    for r in range(2):
        x = ax[r, c]; vw.draw(x, vw.colors(), alpha=0.3)
        if not ts:
            x.text(0.5, 0.5, "not run", ha="center", transform=x.transAxes); continue
        t = ts[0]; sl = PL.streamlines_of(t); A, B = t["a"], t["b"]
        k = np.flatnonzero((t["pair"] == 12) & key["valid"][A] & key["valid"][B] & (a["hemi"][A] == h))
        k = rng.permutation(k)[:800] if r == 0 else k_top
        if r == 0:
            k_top = k; v = np.clip((key["ecc"][A[k]] - lo) / (hi - lo), 0, 1); cm = "viridis"
        else:
            v = np.clip(np.hypot(key["x"][A[k]] - key["x"][B[k]], key["y"][A[k]] - key["y"][B[k]]) / emax, 0, 1); cm = "RdYlBu_r"
        x.add_collection(LineCollection([np.c_[sl[i] @ vw.right, sl[i][:, 2]] for i in k], colors=plt.get_cmap(cm)(v), linewidths=0.4))
        if r == 1: x.text(0.98, 0.98, f"n = {len(k)}\nRSI = {subject_rsi(m if name == "REF" else var, name):.2f}", transform=x.transAxes,
                          fontsize=6, ha="right", va="top")
for r, (cm, vmin, vmax, lab) in enumerate((("viridis", lo, hi, "V1-end eccentricity (deg)"), ("RdYlBu_r", 0, emax, "endpoint error (deg)"))):
    ticks = np.linspace(vmin, vmax, 3).round(2)
    cb = plt.colorbar(plt.cm.ScalarMappable(cmap=cm, norm=plt.Normalize(vmin, vmax)), cax=ax[r, 3].inset_axes([1.06, 0.1, 0.05, 0.8]), ticks=ticks)
    cb.set_label(lab)
    if r == 1: cb.ax.set_yticklabels([f"{t:g}" for t in ticks[:-1]] + [f"≥{ticks[-1]:g}"])
for r in range(2): ax[r, 0].text(-0.1, 1.0, "ab"[r], transform=ax[r, 0].transAxes, fontsize=10, fontweight="bold", va="top")
save(fig, "fig3_bundles")

## Figure 4: path loss against voxel loss (H3)
(a–d) V1–V2 eccentricity ratios on the main clinical-grade protocol: reference, classical CSD, and the two networks
that differ only in the loss. (e) RSI on each clinical-grade protocol, one point per subject, lines from the
voxel-loss to the path-loss network of the same subject; the dashed line is the median reference RSI. (f) Voxel-wise
accuracy against RSI, median and inter-quartile range over subjects, for every method and clinical protocol.

In [ ]:
fig = plt.figure(figsize=(7.2, 6.4)); gs = fig.add_gridspec(2, 4, height_ratios=[1, 1.6], hspace=0.35, wspace=0.75)
for c, (m, name, lab) in enumerate((("msmt_csd", "REF", "reference"), ("csd", MAIN, f"classical CSD, {MAIN}"),
                                    ("voxel", MAIN, f"voxel loss, {MAIN}"), ("path", MAIN, f"path loss, {MAIN}"))):
    x = fig.add_subplot(gs[0, c]); R, _ = pooled(m, name, 12, "ecc"); im = matrix(x, R, lab, "V1-V2", "ecc"); letter(x, "abcd"[c], -0.45)
    if c: x.set_ylabel("")
if im is not None: colorbar(fig, im, x)
bottom = gs[1, :].subgridspec(1, 2, wspace=0.3)
x = fig.add_subplot(bottom[0]); w = 0.25; ref_med = ref[ref.method == "msmt_csd"].rsi_ree_all.median()
allm = pd.concat([prot[prot.method == "csd"], lea[lea.variant.isin(["voxel", "path"])]]).groupby(["sid", "protocol", "variant"]).rsi_ree_all.mean().unstack("variant")
for k, p in enumerate(clin):
    d = allm.xs(p, level="protocol") if p in allm.index.get_level_values("protocol") else pd.DataFrame()
    for j, m in enumerate(("csd", "voxel", "path")):
        if m in d: x.scatter(np.full(len(d), k + (j - 1) * w), d[m], s=10, color=COL(m), zorder=3, label=STYLE[m][1] if k == 0 else None)
    if {"voxel", "path"} <= set(d): x.plot([k, k + w], d[["voxel", "path"]].T, color="0.6", lw=0.5)
x.axhline(ref_med, color="k", ls="--", lw=0.8, label="reference, median"); x.axhline(0, color="k", lw=0.5)
x.set_xticks(range(len(clin)), clin); x.set_xlim(-0.6, len(clin) - 0.4); x.set_xlabel("protocol"); x.set_ylabel("RSI"); letter(x, "e")
x.legend(frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.16))
x = fig.add_subplot(bottom[1]); al = pd.concat([prot[prot.protocol.isin(clin)], lea[lea.protocol.isin(clin)]])
al = al.groupby(["sid", "protocol", "variant"])[["acc", "rsi_ree_all"]].mean().reset_index()
order = [m for m in ("csd", "voxel", "path") if m in set(al.variant)] + sorted(set(al.variant) - {"csd", "voxel", "path"})
marks = dict(zip(clin, "os^dv<>"))
for (p, m), d in al.groupby(["protocol", "variant"]):
    q = d[["acc", "rsi_ree_all"]].quantile([.25, .5, .75])
    x.errorbar(q.acc[.5], q.rsi_ree_all[.5], xerr=[[q.acc[.5] - q.acc[.25]], [q.acc[.75] - q.acc[.5]]],
               yerr=[[q.rsi_ree_all[.5] - q.rsi_ree_all[.25]], [q.rsi_ree_all[.75] - q.rsi_ree_all[.5]]],
               fmt=marks[p], color=COL(m), ms=4, lw=0.6, capsize=0, zorder=3 if m in ("voxel", "path") else 2)
x.axhline(ref_med, color="k", ls="--", lw=0.8); x.set_xlabel("voxel-wise accuracy (ACC)"); x.set_ylabel("RSI"); letter(x, "f")
h = [plt.Line2D([], [], color=COL(m), marker="o", ls="", ms=4, label=STYLE[m][1] if m in STYLE else m) for m in order]
h += [plt.Line2D([], [], color="0.3", marker=marks[p], ls="", ms=4, label=p) for p in clin]
x.legend(handles=h, frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.16), columnspacing=0.8, handletextpad=0.2)
save(fig, "fig4_h3_path_vs_voxel")

## Figure 5: recovery of voxel-wise accuracy and of retinotopy (H2)
Recovery = (network − classical CSD) / (reference − classical CSD) on the same protocol, for the voxel-loss network,
as in the H2 test of notebook 09: cells where CSD loses less than 0.02 RSI are left out. (a) One small point per
subject × protocol, one large point per subject (median over protocols); above the line, accuracy recovers more.
(b) The same for the path-loss network, for comparison.

In [ ]:
def recovery(var):
    lo = prot[prot.method == "csd"].set_index(["sid", "protocol"])[["acc", "rsi_ree_all"]]
    net = lea[lea.variant == var].groupby(["sid", "protocol"])[["acc", "rsi_ree_all"]].mean()
    j = net.join(lo, rsuffix="_csd", how="inner").reset_index(); j = j[j.protocol.isin(clin)]
    j["rsi_ref"] = j.sid.map(ref[ref.method == "msmt_csd"].set_index("sid").rsi_ree_all)
    lost = j.rsi_ref - j.rsi_ree_all_csd
    j["rec_acc"] = (j.acc - j.acc_csd) / (1 - j.acc_csd); j["rec_rsi"] = ((j.rsi_ree_all - j.rsi_ree_all_csd) / lost).where(lost >= 0.02)
    return j.replace([np.inf, -np.inf], np.nan)

fig, ax = plt.subplots(1, 2, figsize=(7.2, 3.4)); rec = {v: recovery(v) for v in ("voxel", "path")}
v = np.r_[[0, 1], *[np.r_[j.rec_rsi.dropna(), j.rec_acc.dropna()] for j in rec.values()]]; lim = [v.min() - 0.1 * np.ptp(v), v.max() + 0.1 * np.ptp(v)]
for x, var, s in zip(ax, ("voxel", "path"), "ab"):
    j = rec[var]; med = j.groupby("sid")[["rec_acc", "rec_rsi"]].median()
    x.scatter(j.rec_rsi, j.rec_acc, s=6, color=COL(var), alpha=0.4); x.scatter(med.rec_rsi, med.rec_acc, s=24, color=COL(var), edgecolor="k", lw=0.4)
    x.plot(lim, lim, color="k", lw=0.6, ls=":"); x.axhline(0, color="0.8", lw=0.5); x.axvline(0, color="0.8", lw=0.5)
    x.set_xlim(lim); x.set_ylim(lim); x.set_aspect("equal"); x.set_xlabel("recovery of RSI"); x.set_title(STYLE[var][1]); letter(x, s)
    x.text(0.98, 0.02, f"{int(j.rec_rsi.isna().sum())} of {len(j)} cells without RSI loss", transform=x.transAxes, ha="right", fontsize=6)
ax[0].set_ylabel("recovery of voxel-wise accuracy"); save(fig, "fig5_h2_recovery")

## Figure 6: other scanners and reproducibility
RSI per subject on (a) HCP 7T and (b) NSD scans, which no network saw in training; lines join the voxel-loss and
path-loss networks of a subject. (c) Agreement of the reference tractogram between two scans of the same person
(HCP test and retest; NSD run 1 and run 2): weighted Dice and the correlation of streamline-end counts per vertex.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(7.2, 2.8), gridspec_kw=dict(width_ratios=[1, 1, 0.8], wspace=0.45))
for x, (f, lab), s in zip(ax, (("external_7t", "HCP 7T"), ("external_nsd", "NSD")), "ab"):
    d = table(f)
    if d.empty: x.text(0.5, 0.5, "no data", ha="center", transform=x.transAxes); x.set_title(lab); continue
    w = d.groupby(["sid", "variant"]).rsi_ree_all.mean().unstack(); order = [m for m in ("proximity", "csd", "voxel", "path") if m in w] + sorted(set(w) - set(STYLE))
    for k, m in enumerate(order): x.scatter(np.full(len(w), k), w[m], s=10, color=COL(m), zorder=3)
    if {"voxel", "path"} <= set(w): x.plot([order.index("voxel"), order.index("path")], w[["voxel", "path"]].T, color="0.6", lw=0.5)
    x.axhline(0, color="k", lw=0.5); x.set_xticks(range(len(order)), order, rotation=45, ha="right"); x.set_ylabel("RSI"); x.set_title(f"{lab} (n = {len(w)})"); letter(x, s)
x = ax[2]; rows = []
for f, lab in (("retest_overlap", "HCP retest"), ("nsd_run_overlap", "NSD runs")):
    d = table(f)
    if not d.empty: rows.append(d[d.method == d.method.iloc[0]].assign(dataset=lab))
if rows:
    d = pd.concat(rows)
    for k, (col, lab) in enumerate((("wdice_retest", "weighted Dice"), ("vertex_count_r", "end-count r"))):
        for j, (ds, g) in enumerate(d.groupby("dataset", sort=False)):
            x.scatter(np.full(len(g), k + (j - 0.5) * 0.3), g[col], s=12, color=f"C{j + 4}", label=ds if k == 0 else None)
    x.set_xticks([0, 1], ["weighted Dice", "end-count r"]); x.set_ylim(0, 1); x.legend(frameon=False, fontsize=6)
x.set_title("reproducibility"); letter(x, "c")
save(fig, "fig6_external_reproducibility")

## Supplementary figures
S1: every learned variant and classical method on the clinical-grade protocols (RSI and voxel-wise accuracy).
S2: reference RSI under the alternative analysis choices of notebook 08. S3: the path-against-voxel contrast scored
with keys that need no functional scan (group-average pRF maps, anatomical atlas). S4: all pairwise comparisons.

In [ ]:
al = pd.concat([prot[prot.protocol.isin(clin)], lea[lea.protocol.isin(clin)]]).groupby(["sid", "protocol", "variant"])[["rsi_ree_all", "acc"]].mean().reset_index()
order = [m for m in ("csd", "voxel", "path") if m in set(al.variant)] + sorted(set(al.variant) - {"csd", "voxel", "path"})
fig, ax = plt.subplots(2, 1, figsize=(7.2, 4.6), sharex=True)
for x, col, lab in zip(ax, ("rsi_ree_all", "acc"), ("RSI", "voxel-wise accuracy (ACC)")):
    for k, m in enumerate(order):
        for j, p in enumerate(clin):
            v = al[(al.variant == m) & (al.protocol == p)][col].dropna(); dx = (j - (len(clin) - 1) / 2) * 0.22
            if len(v): x.boxplot(v, positions=[k + dx], widths=0.18, showfliers=False, patch_artist=True,
                                 boxprops=dict(facecolor=f"C{j}", alpha=0.35, lw=0.5), medianprops=dict(color="k", lw=0.8), whiskerprops=dict(lw=0.5), capprops=dict(lw=0.5))
            x.scatter(np.full(len(v), k + dx), v, s=4, color=f"C{j}", zorder=3)
    x.set_ylabel(lab)
for j, p in enumerate(clin): ax[0].scatter([], [], color=f"C{j}", s=12, label=p)
ax[0].legend(frameon=False, ncol=len(clin), loc="lower right", bbox_to_anchor=(1, 1.0)); ax[0].axhline(ref[ref.method == "msmt_csd"].rsi_ree_all.median(), color="k", ls="--", lw=0.8)
ax[1].set_xticks(range(len(order)), order, rotation=30, ha="right"); letter(ax[0], "a"); letter(ax[1], "b")
save(fig, "figS1_all_methods")

rb = table("robustness")
if not rb.empty:
    fig, x = plt.subplots(figsize=(7.2, 2.6)); an = list(dict.fromkeys(rb.analysis))
    for k, n in enumerate(an):
        v = rb[rb.analysis == n].rsi_ree_all.dropna(); x.scatter(np.full(len(v), k), v, s=10, color="k" if n == "default" else "C0")
        if len(v): x.hlines(v.median(), k - 0.3, k + 0.3, color="C3", lw=1.2)
    x.axhline(0, color="k", lw=0.5); x.set_xticks(range(len(an)), an, rotation=35, ha="right"); x.set_ylabel("reference RSI")
    save(fig, "figS2_robustness")

fig, ax = plt.subplots(1, 3, figsize=(7.2, 2.6), sharey=True)
for x, (col, lab) in zip(ax, (("rsi_ree_all", "measured pRF"), ("template_rsi_ree_all", "group-average pRF"), ("atlas_rsi_ree_all", "anatomical atlas"))):
    w = lea[lea.protocol.isin(clin)].groupby(["sid", "protocol", "variant"])[col].mean().unstack()
    if {"voxel", "path"} <= set(w):
        for j, p in enumerate(clin):
            if p in w.index.get_level_values("protocol"):
                d = w.xs(p, level="protocol"); x.plot([j - 0.12, j + 0.12], d[["voxel", "path"]].T, color="0.6", lw=0.5)
                x.scatter(np.full(len(d), j - 0.12), d.voxel, s=8, color=COL("voxel")); x.scatter(np.full(len(d), j + 0.12), d.path, s=8, color=COL("path"))
    x.axhline(0, color="k", lw=0.5); x.set_xticks(range(len(clin)), clin); x.set_title(f"key: {lab}")
ax[0].set_ylabel("RSI")
ax[1].legend(handles=[plt.Line2D([], [], color=COL(m), marker="o", ls="", ms=4, label=STYLE[m][1]) for m in ("voxel", "path")],
             frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.12))
save(fig, "figS3_keys")

### S4: pairwise comparisons
FDR-corrected p-values of the two-sided Wilcoxon test of RSI between every pair of methods on each clinical-grade
protocol (notebook 09, `pairwise_tests.csv`). Green below `stats.alpha`; grey where the test needs more subjects.

In [ ]:
from matplotlib.colors import BoundaryNorm, LinearSegmentedColormap
pw = table("pairwise_tests")
if not pw.empty:
    label = lambda m: f"{m[4:]} (network)" if m.startswith("net_") else m
    green = LinearSegmentedColormap.from_list("p", ["#00441b", "#41ab5d", "#c7e9c0", "#f7fcf5"])
    edges = [0, 0.001, 0.01, cfg.stats.alpha, 1]; norm = BoundaryNorm(edges, green.N)
    ps = list(dict.fromkeys(pw.protocol)); n = len(ps)
    fig, ax = plt.subplots(1, n, figsize=(7.2, 7.2 / n + 1.2), squeeze=False); ax = ax[0]
    for x, p in zip(ax, ps):
        d = pw[pw.protocol == p]; ms = list(dict.fromkeys(list(d.method_1) + list(d.method_2))); k = len(ms)
        P = np.full((k, k), np.nan)
        for _, row in d.iterrows(): P[ms.index(row.method_2), ms.index(row.method_1)] = row.p_fdr
        low = np.tril(np.ones((k, k), bool), -1)
        x.imshow(np.where(low & np.isnan(P), 1.0, np.nan), cmap="Greys", vmin=0, vmax=4)          # untested cells
        im = x.imshow(np.where(low, P, np.nan), cmap=green, norm=norm)
        for i, j in zip(*np.nonzero(low)):
            if np.isfinite(P[i, j]): x.text(j, i, "<.001" if P[i, j] < 0.001 else f"{P[i, j]:.3f}".lstrip("0"), ha="center", va="center",
                                            fontsize=4.5, color="w" if P[i, j] < 0.01 else "k")
        x.set_xticks(range(k - 1), [label(m) for m in ms[:-1]], rotation=90); x.set_yticks(range(1, k), [label(m) for m in ms[1:]])
        x.set_xlim(-0.5, k - 1.5); x.set_ylim(k - 0.5, 0.5); x.set_title(p); x.tick_params(length=0, labelsize=5)
        for sp in x.spines.values(): sp.set_visible(False)
    cb = fig.colorbar(im, ax=list(ax), shrink=0.6, ticks=edges, pad=0.02); cb.set_label("FDR-corrected p"); cb.ax.tick_params(labelsize=5)
    save(fig, "figS4_pairwise")
else:
    print("pairwise_tests.csv not found: run notebook 09 first")